In [1]:
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq

The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum fonts-ipafont-gothic
  fonts-ipafont-mincho fonts-wqy-microhei fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 29 not upgraded.
Need to get 39.7 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 124947 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .../openjdk-8-jre-headless_8u442-b06~us1-0ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-8-jre-headless:amd64 (8u442-b06~us1-0

In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [3]:
conf = SparkConf().set("spark.ui.port", "4050")

In [4]:
import pyspark
spark = SparkSession.builder.getOrCreate()

In [6]:
from google.colab import files

uploaded = files.upload()  # This will prompt you to upload a file


Saving error_rankings.csv to error_rankings.csv


In [7]:
from google.colab import files

uploaded = files.upload()  # This will prompt you to upload a file


Saving user_logs.csv to user_logs.csv


In [10]:
from pyspark.sql.functions import col, sum, count, when

In [9]:
log_data = spark.read.csv("user_logs.csv", header=True, inferSchema=True)
error_data = spark.read.csv("error_rankings.csv", header=True, inferSchema=True)

In [16]:
# getting highest Rate records (Rule 1 : User is flagged when User.errorRate > 50)

In [11]:
highErrorRate = log_data.groupBy("Username").agg(
    sum(when(col("LogType")=="ERROR", col("Count")).otherwise(0)).alias("ErrorCount"),
    sum(col("Count")).alias("TotalCount")).withColumn("ErrorRate", col("ErrorCount") / col("TotalCount")*100)

In [14]:
higherErrorRate = highErrorRate.filter(col("ErrorRate") > 50)

In [15]:
higherErrorRate.select("Username","ErrorRate").show()

+--------+------------------+
|Username|         ErrorRate|
+--------+------------------+
|   breee|52.083333333333336|
+--------+------------------+



In [17]:
# checking for System wide issue. (Rule 2: Most common error >= 2.5* Second Most common error.)

In [18]:
topErrors = error_data.orderBy(col("Count").desc())
mostCommonError = topErrors.first()
secondMostCommonError = topErrors.collect()[1]

In [24]:
if mostCommonError[1]/secondMostCommonError[1] >= 2.5:
    print(f"System wide issue '{mostCommonError[0]}' occurs {mostCommonError[1]} times")
else:
    print("No system wide issue")

No system wide issue


In [25]:
# Finding Rare Errors (Rule 3: Rare errors could signal serious issues that might cause failures or crashes.)

In [29]:
rareErrors = error_data.filter(col("count")<5).select("Error Message", "Count").show()

+-------------+-----+
|Error Message|Count|
+-------------+-----+
+-------------+-----+



In [30]:
# Finding User with 0 INFO  (Rule 4: User who has only Error)

In [32]:
onlyErrors = log_data.filter("LogType ='INFO' AND Count = 0").select("Username").show()

+--------+
|Username|
+--------+
+--------+



In [33]:
print("Anomaly Detection Completed! ")

Anomaly Detection Completed! 
